In [1]:
import kaleidocell
import scanpy as sc
import os


In [2]:
adata = sc.read_h5ad("/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/data/01_extGBmap/immune_subsets/TAM-BDM.h5ad")


In [3]:
print(adata.obs.columns.tolist())
print(adata.obs["donor_id"].nunique())
print(f"{adata.obs['annotation_level_3'].unique()[0]}: {adata.n_obs} cells")

['author', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'annotation_level_1', 'annotation_level_2', 'annotation_level_3', 'gbmap', 'suspension_type', 'stage', 'location', 'sector', 'celltype_original', 'EGFR', 'MET', 'p53', 'TERT', 'ATRX', 'PTEN', 'MGMT', 'chr1p19q', 'PDGFR', 'tissue_ontology_term_id', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid']
228
TAM-BDM: 267733 cells


In [4]:
results_nmf, nmf_convergence = kaleidocell.multi_sample_nmf(
    adata,
    batch_key="donor_id",
    test_ranks=[4, 5, 6, 7, 8, 9],
    n_initializations=1,
    max_iterations=100
)

Running multi-sample NMF on 228 samples.
Tested ranks: [4, 5, 6, 7, 8, 9]


Global NMF Progress:   0%|          | 0/136800 [00:00<?, ?it/s]

/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/nmf.py:104: UserWarning: Number of rows/columns is smaller than rank; results may be unstable.
  warnings.warn(
/net/bq-storage/ag-cherrmann/bq_cinac/pr

Multi-sample NMF complete.


In [5]:
results_mp = kaleidocell.derive_nmf_metaprograms(results_nmf)
print(results_mp["metrics"])

Total number of programs: 8892
Optimal number of meta-programs: 18
Clustering using hierarchical Ward linkage
Defining meta-program consensus
Computing meta-program metrics
Meta-program derivation complete.
      sampleCoverage  silhouette  meanSimilarity  nPrograms  nGenes
MP1         0.504386    0.085279           0.221        592      29
MP2         0.758772   -0.034589           0.038       3675    9246
MP3         0.289474    0.193273           0.286        230     113
MP4         0.043860    0.023085           0.110        245     282
MP5         0.425439   -0.047715           0.168        496     768
MP6         0.157895    0.332362           0.434         72       3
MP7         0.359649    0.058625           0.255        243      23
MP8         0.377193   -0.014542           0.140        284       7
MP9         0.153509    0.486035           0.561         63       1
MP10        0.078947    0.379079           0.498         57      32
MP11        0.320175    0.355587           0.

In [6]:
mp_scores = kaleidocell.compute_mp_scores(results_mp, adata)

Computing module scores per MP.
Finished computing MP scores.


In [7]:
path = kaleidocell.get_html(
    results_mp,
    adata,
    mp_scores=mp_scores,
    obs=["stage", "donor_id"],
    output_path="/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_TAM-BDM"
)
print(f"Report written to: {path}")

Rendering heatmap…
Rendering UMAP scores…
Building metrics table…
Running GSEA — GO Biological Process…
Ensembl gene IDs detected — translating to HGNC using bundled table.
Building gene table…
Rendering violin plots for 'stage'…
Plotting MP distributions across 'stage'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.
Rendering violin plots for 'donor_id'…
Plotting MP distributions across 'donor_id'.


/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.violinplot(data=df, x=batch_key, y=score_col, palette=palette, inner="box")
/net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/.environments/kaleidocell_env/lib/python3.11/site-packages/kaleidocell/visualization.py:360: FutureWarning: 

Pas

Finished plotting MP distributions.

Files written to /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_TAM-BDM/
  results.html
  gsea_GO_Biological_Process.csv
  genes.csv
Report written to: /net/bq-storage/ag-cherrmann/bq_cinac/projects/synimmune/results/02_NMF_immune_subsets/02_nmf_TAM-BDM/results.html
